# Tema: Lakeflow Spark Declarative Pipelines

## Objetivos
Construir Bronze→Silver→Gold con streaming tables y materialized views; practicar AUTO CDC.

## Conceptos importantes para el examen
pyspark.pipelines como dp; funciones declarativas sin efectos laterales; DAG inferido; destinos gobernados; AUTO CDC con sequence_by.

**Dificultad:** Intermedio · **Tiempo estimado:** 90 min.

Este notebook prepara datos y comprueba resultados. La fuente real es resources/pipelines/medallion.py: añádela a un pipeline desde la interfaz, configura lab.source_table con el valor impreso y el catálogo/schema destino. Usa una edición con expectations. Ejecuta una actualización Triggered. No ejecutes las funciones dp en cómputo interactivo.

Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_22_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
pipeline_input = spark.createDataFrame([(i, i % 4, float(i*10) if i != 12 else -5.0) for i in range(1,13)], "order_id INT, customer_id INT, amount DOUBLE")
pipeline_input.write.format("delta").mode("errorifexists").saveAsTable("pipeline_input")
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_input"
PIPELINE_SCHEMA = SCHEMA + "_pipeline"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(PIPELINE_SCHEMA)}")
print("lab.source_table =", SOURCE_TABLE)
print("Catálogo destino =", CATALOG, "Schema destino =", PIPELINE_SCHEMA)
RUN_PIPELINE_CHECKS = False  # Cambiar a True después de ejecutar el pipeline real.

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Contraste batch local
Esto permite probar la transformación, pero no ejecuta el servicio Lakeflow.

In [ ]:
valid = pipeline_input.filter("amount >= 0")
display(valid.groupBy("customer_id").agg(F.sum("amount").alias("revenue")))
assert valid.count() == 11

### 2. Verificar pipeline real
En Jobs & Pipelines crea pipeline ETL, añade medallion.py, indica destino y lab.source_table. La identidad del pipeline necesita SELECT en fuente y creación en destino. Inicia Update y observa el grafo.

In [ ]:
def pipeline_table(name):
    return f"{ident(CATALOG)}.{ident(PIPELINE_SCHEMA)}.{ident(name)}"
if RUN_PIPELINE_CHECKS:
    display(spark.table(pipeline_table("gold_customers")))
    assert spark.table(pipeline_table("silver_orders")).count() == 11
else:
    print("Pendiente de ejecutar pipeline. La comparación batch no cuenta como ejecución Lakeflow.")

### 3. Preparar CDC independiente
Para AUTO CDC crea otro pipeline con solo auto_cdc.py y lab.cdc_source. Requiere la edición/funcionalidad de CDC habilitada.

In [ ]:
change_events = spark.createDataFrame(
 [(i, f"Cliente {i:02d}", "Madrid", datetime(2026,1,1), i, "UPSERT") for i in range(1,13)] +
 [(2, "Cliente 02", "Bilbao", datetime(2026,2,1), 20, "UPSERT"),
  (2, "Cliente 02", "Cádiz", datetime(2026,1,15), 15, "UPSERT")],
 "customer_id INT, name STRING, city STRING, updated_at TIMESTAMP, event_seq LONG, op STRING")
change_events.write.format("delta").mode("errorifexists").saveAsTable("customer_change_events")
print("lab.cdc_source =", f"{CATALOG}.{SCHEMA}.customer_change_events")

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Ejecuta medallion.py en un pipeline y verifica 12 Bronze, 11 Silver y revenue total 660. Si no tienes servicio, calcula las mismas métricas batch y marca Lakeflow pendiente.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Añade un pedido 13 de 50 y actualiza el pipeline. Comprueba total Gold 710.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Añade una materialized view gold_totals con ingresos globales. Guarda la definición como fuente del pipeline y ejecuta Update.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Configura AUTO CDC con auto_cdc.py en otro schema de destino. Identifica ciudad actual e historial del cliente 2.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Añade un DELETE del cliente 3 al origen CDC y ejecuta otra actualización. Comprueba que no tiene versión vigente.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** La fila con order_id 12 es negativa.

**Pista 2:** Añade a la fuente; no escribas en la streaming table gestionada.

**Pista 3:** Una MV usa lectura batch del objeto Silver.

**Pista 4:** __END_AT nulo indica versión vigente; SEQUENCE BY usa fecha y secuencia.

**Pista 5:** apply_as_deletes interpreta op; no borra el historial SCD2.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
if RUN_PIPELINE_CHECKS:
    assert spark.table(pipeline_table("bronze_orders")).count() == 12
    assert spark.table(pipeline_table("silver_orders")).count() == 11
    assert spark.table(pipeline_table("gold_customers")).agg(F.sum("revenue")).first()[0] == 660
else:
    assert pipeline_input.filter("amount>=0").agg(F.sum("amount")).first()[0] == 660

### Solución 2

In [ ]:
spark.sql("INSERT INTO pipeline_input VALUES (13, 1, 50.0)")
print("Ejecuta Update en el pipeline antes de la siguiente comprobación.")
# Ejecutar en otra celda después del Update:
# assert spark.table(pipeline_table("gold_customers")).agg(F.sum("revenue")).first()[0] == 710

### Solución 3

In [ ]:
definition = '''from pyspark import pipelines as dp
from pyspark.sql import functions as F
@dp.materialized_view(name="gold_totals")
def gold_totals():
    return spark.read.table("silver_orders").agg(F.sum("amount").alias("revenue"))
'''
print(definition)
# Añadir este código al archivo fuente del pipeline; no ejecutar aquí.
# Verificación después de Update:
if RUN_PIPELINE_CHECKS:
    display(spark.table(pipeline_table("gold_customers")))

### Solución 4

In [ ]:
CDC_SCHEMA = SCHEMA + "_cdc"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(CDC_SCHEMA)}")
print("Destino del pipeline AUTO CDC:", f"{CATALOG}.{CDC_SCHEMA}")
print("Fuente:", f"{CATALOG}.{SCHEMA}.customer_change_events")
RUN_CDC_CHECKS = False
if RUN_CDC_CHECKS:
    dim = spark.table(f"{ident(CATALOG)}.{ident(CDC_SCHEMA)}.dim_customers_scd2")
    display(dim.filter("customer_id=2").orderBy("__START_AT"))
    assert dim.filter("customer_id=2 AND __END_AT IS NULL").first().city == "Bilbao"
else:
    # Simulación batch del orden de negocio; no sustituye ejecución AUTO CDC.
    display(change_events.filter("customer_id=2").orderBy("updated_at","event_seq"))

### Solución 5

In [ ]:
spark.sql("INSERT INTO customer_change_events VALUES (3, 'Cliente 03', 'Madrid', TIMESTAMP '2026-03-01', 30, 'DELETE')")
print("Después de Update del pipeline AUTO CDC:")
print(f"SELECT * FROM {CATALOG}.{SCHEMA}_cdc.dim_customers_scd2 WHERE customer_id=3 AND __END_AT IS NULL;")
# Resultado esperado: cero filas vigentes.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué devuelve una función declarativa de dataset?

A. Una escritura saveAsTable ejecutada manualmente

B. Un DataFrame

C. Un token de acceso

D. Un gráfico HTML

### Pregunta 2
¿Qué decorador expresa resultados batch persistidos?

A. @dp.table siempre

B. @dp.expect

C. @dp.materialized_view

D. @staticmethod

### Pregunta 3
¿Qué ordena cambios fuera de orden en AUTO CDC?

A. sequence_by

B. El número de workers

C. El orden alfabético del cliente

D. El nombre del notebook

### Respuestas y explicación
**1. B** — El motor interpreta la definición y gestiona la ejecución.

**2. C** — Declara una materialized view.

**3. A** — La secuencia representa el orden lógico de los eventos.

### Documentación oficial
- [API Python](https://docs.databricks.com/aws/en/ldp/developer/python-dev)
- [AUTO CDC](https://docs.databricks.com/aws/en/ldp/developer/ldp-python-ref-apply-changes)

## PARTE 6 - RETO FINAL
Amplía el pipeline con cuarentena y un resumen por ciudad usando una dimensión. Compara la actualización de una MV con una copia CTAS y guarda evidencias del DAG.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
